In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

source_table = "dbr_dev_ua5816bd.gulidov_artem_silver.products_silver"
#demo_table is an exact 1:1 copy of the silver table used here as a test dummy to show all the relevant schema and table-realated stuff
demo_table = "dbr_dev_ua5816bd.gulidov_artem_silver.products_silver_demo_schema"


##Schema enforcement test

In [0]:
test_df = (
    spark.table(demo_table)
    .limit(1)
    .withColumn("surprise_column", F.lit("alkfjasdklfj;lakdsj"))
)

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")

try:
    (
        test_df.write 
        .format("delta")
        .mode("append")
        .option("mergeSchema", "false")
        .saveAsTable(demo_table)
    )
except Exception as e:
    print("Write rejected:" + str(e))

##Schema evolution test

In [0]:
test_df = (
    spark.table(demo_table)
    .limit(1)
    .withColumn("surprise_column_2", F.lit("alkfjasdklfj;lakdsj"))
)

try:
    (
        test_df.write 
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(demo_table)
    )
except Exception as e:
    print("Write rejected:" + str(e))

##Table maintenance and similar

In [0]:
optimize_result = spark.sql(f"""
OPTIMIZE {demo_table}
""")

spark.sql(f"""
VACUUM {demo_table}
""")


#delta column mapping
spark.sql(f"""
ALTER TABLE {demo_table}
SET TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name'
);
""")

spark.sql(f"""
ALTER TABLE {demo_table}
RENAME COLUMN entry_creator TO creator;
""")

assert "creator" in spark.table(demo_table).columns
assert "entry_creator" not in spark.table(demo_table).columns